In [1]:
import numpy as np
from typing import Dict, List

Identificador de BSPs

In [2]:
def identificador(tablero, color_jugador):

    columnas = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
    filas = ['1', '2', '3', '4', '5', '6', '7', '8']
    
    # color_jugador: 1 = negro, -1 = blanco (estándar OthelloBoardState)
    mi_valor = color_jugador
    oponente_valor = -color_jugador
    
    bsps = {}
    
    # ----- Identificar estado de cada casilla -----
    for fila in range(8):
        for col in range(8):
            casilla = columnas[col] + filas[fila]
            valor = tablero[fila][col]
            
            # BSP: Casilla vacía (0), mía (1), oponente (2)
            bsps[f"BSP{casilla}0"] = (valor == 0)
            bsps[f"BSP{casilla}1"] = (valor == mi_valor)
            bsps[f"BSP{casilla}2"] = (valor == oponente_valor)
    
    # ----- Contar fichas -----
    total = sum(1 for fila in tablero for v in fila if v != 0)
    mis_fichas = sum(1 for fila in tablero for v in fila if v == mi_valor)
    fichas_op = sum(1 for fila in tablero for v in fila if v == oponente_valor)
    
    # ----- Propiedades especiales -----
    
    # Estado inicial: 4 fichas en el centro
    # OthelloBoardState: [3,4]=1(negro), [3,3]=-1(blanco), [4,3]=1(negro), [4,4]=-1(blanco)
    estado_inicial = (
        total == 4 and
        tablero[3][3] == -1 and  # D4 blanca
        tablero[3][4] == 1 and   # E4 negra
        tablero[4][3] == 1 and   # D5 negra
        tablero[4][4] == -1      # E5 blanca
    )
    
    # Partida terminada: tablero lleno
    terminada = (total == 64)
    
    bsps["BSP_INICIAL"] = estado_inicial
    bsps["BSP_TERMINADA"] = terminada
    bsps["BSP_GANE"] = terminada and mis_fichas > fichas_op
    bsps["BSP_PERDI"] = terminada and mis_fichas < fichas_op
    bsps["BSP_EMPATE"] = terminada and mis_fichas == fichas_op
    bsps["BSP_EN_CURSO"] = not terminada
    
    return bsps

Funciones auxiliares

In [ ]:
def imprimir_tablero(tablero):
    
    print("  A B C D E F G H")
    for i, fila in enumerate(tablero):
        print(f"{i+1}", end=" ")
        for valor in fila:
            if valor == 0:
                print(".", end=" ")
            elif valor == 1:
                print("●", end=" ")  # Negra
            else:
                print("○", end=" ")  # Blanca
        print(f"{i+1}")
    print("  A B C D E F G H")


def obtener_bsps_activas(bsps):
    """Retorna solo las BSPs que son True"""
    return {k: v for k, v in bsps.items() if v}


def contar_bsps_por_tipo(bsps):
    """Cuenta cuántas BSPs de cada tipo están activas"""
    vacias = sum(1 for k, v in bsps.items() if v and k.endswith('0') and 'BSP_' not in k)
    mias = sum(1 for k, v in bsps.items() if v and k.endswith('1'))
    oponente = sum(1 for k, v in bsps.items() if v and k.endswith('2'))
    
    return {
        "vacias": vacias,
        "mias": mias,
        "oponente": oponente
    }


def imprimir_resumen(bsps):
    """Imprime un resumen de las BSPs"""
    print("=" * 70)
    print("RESUMEN DE BSPs")
    print("=" * 70)
    
    conteo = contar_bsps_por_tipo(bsps)
    print(f"Casillas vacías: {conteo['vacias']}")
    print(f"Casillas mías: {conteo['mias']}")
    print(f"Casillas del oponente: {conteo['oponente']}")
    
    print("\nPropiedades especiales:")
    if bsps.get("BSP_INICIAL"):
        print("  ✓ Estado inicial")
    if bsps.get("BSP_TERMINADA"):
        print("  ✓ Partida terminada")
    if bsps.get("BSP_GANE"):
        print("  ✓ Posición ganada")
    if bsps.get("BSP_PERDI"):
        print("  ✓ Posición perdida")
    if bsps.get("BSP_EMPATE"):
        print("  ✓ Posición empatada")
    if bsps.get("BSP_EN_CURSO"):
        print("  ✓ Juego en curso")
    
    print("=" * 70)

In [ ]:
def imprimir_todas_bsps_activas(bsps):
    """Imprime TODAS las BSPs que son True (incluyendo vacias)"""
    print("\n" + "=" * 70)
    print("TODAS LAS BSPs ACTIVAS (True)")
    print("=" * 70)
    
    # Contar cuantas hay
    activas = [k for k, v in bsps.items() if v]
    print(f"\nTotal: {len(activas)} BSPs activas\n")
    
    # Separar por tipo
    casillas_vacias = []
    casillas_mias = []
    casillas_oponente = []
    propiedades_especiales = []
    
    for bsp_id, valor in bsps.items():
        if valor:
            if 'BSP_' in bsp_id:
                propiedades_especiales.append(bsp_id)
            elif bsp_id.endswith('0'):
                casillas_vacias.append(bsp_id)
            elif bsp_id.endswith('1'):
                casillas_mias.append(bsp_id)
            elif bsp_id.endswith('2'):
                casillas_oponente.append(bsp_id)
    
    # Imprimir casillas vacias
    print(f"Casillas vacias ({len(casillas_vacias)}):")
    for bsp_id in casillas_vacias:
        casilla = bsp_id[3:-1]
        print(f"  {bsp_id}: {casilla} vacia")
    
    # Imprimir fichas mias
    print(f"\nFichas mias ({len(casillas_mias)}):")
    for bsp_id in casillas_mias:
        casilla = bsp_id[3:-1]
        print(f"  {bsp_id}: Ficha mia en {casilla}")
    
    # Imprimir fichas oponente
    print(f"\nFichas del oponente ({len(casillas_oponente)}):")
    for bsp_id in casillas_oponente:
        casilla = bsp_id[3:-1]
        print(f"  {bsp_id}: Ficha del oponente en {casilla}")
    
    # Imprimir propiedades especiales
    print(f"\nPropiedades especiales ({len(propiedades_especiales)}):")
    for bsp_id in propiedades_especiales:
        print(f"  {bsp_id}: True")
    
    print("=" * 70)


def imprimir_bsps_activas_con_fichas(bsps):
    """Imprime solo las BSPs activas que NO son vacias (tienen fichas)"""
    print("\n" + "=" * 70)
    print("BSPs ACTIVAS CON FICHAS (sin vacias)")
    print("=" * 70)
    
    # Separar por tipo
    casillas_mias = []
    casillas_oponente = []
    propiedades_especiales = []
    
    for bsp_id, valor in bsps.items():
        if valor:
            if 'BSP_' in bsp_id:
                propiedades_especiales.append(bsp_id)
            elif bsp_id.endswith('1'):
                casillas_mias.append(bsp_id)
            elif bsp_id.endswith('2'):
                casillas_oponente.append(bsp_id)
    
    # Contar total (sin vacias)
    total_con_fichas = len(casillas_mias) + len(casillas_oponente) + len(propiedades_especiales)
    print(f"\nTotal: {total_con_fichas} BSPs activas (sin contar vacias)\n")
    
    # Imprimir fichas mias
    print(f"Fichas mias ({len(casillas_mias)}):")
    for bsp_id in casillas_mias:
        casilla = bsp_id[3:-1]
        print(f"  {bsp_id}: Ficha mia en {casilla}")
    
    # Imprimir fichas oponente
    print(f"\nFichas del oponente ({len(casillas_oponente)}):")
    for bsp_id in casillas_oponente:
        casilla = bsp_id[3:-1]
        print(f"  {bsp_id}: Ficha del oponente en {casilla}")
    
    # Imprimir propiedades especiales
    print(f"\nPropiedades especiales ({len(propiedades_especiales)}):")
    for bsp_id in propiedades_especiales:
        print(f"  {bsp_id}: True")
    
    print("=" * 70)

Ejemplo 1: Tablero inicial


In [ ]:
# Crear tablero en estado inicial
# Estándar OthelloBoardState: 1 = negro, -1 = blanco
tablero_inicial = [
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, -1, 1, 0, 0, 0],  # D4=blanca(-1), E4=negra(1)
    [0, 0, 0, 1, -1, 0, 0, 0],  # D5=negra(1), E5=blanca(-1)
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0]
]

print("TABLERO INICIAL:")
imprimir_tablero(tablero_inicial)

# Identificar BSPs desde perspectiva de negras (color_jugador=1)
bsps = identificador(tablero_inicial, color_jugador=1)

print(f"\nTotal de BSPs: {len(bsps)}")
print(f"BSPs activas (True): {sum(bsps.values())}")

# Imprimir TODAS las BSPs activas (incluyendo vacias)
imprimir_todas_bsps_activas(bsps)

# Imprimir solo BSPs activas con fichas (sin vacias)
imprimir_bsps_activas_con_fichas(bsps)

# Mostrar resumen
print()
imprimir_resumen(bsps)

TABLERO INICIAL:
  A B C D E F G H
1 . . . . . . . . 1
2 . . . . . . . . 2
3 . . . . . . . . 3
4 . . . ○ ● . . . 4
5 . . . ● ○ . . . 5
6 . . . . . . . . 6
7 . . . . . . . . 7
8 . . . . . . . . 8
  A B C D E F G H

Total de BSPs: 198
BSPs activas (True): 66

TODAS LAS BSPs ACTIVAS (True)

Total: 66 BSPs activas

Casillas vacias (60):
  BSPA10: A1 vacia
  BSPB10: B1 vacia
  BSPC10: C1 vacia
  BSPD10: D1 vacia
  BSPE10: E1 vacia
  BSPF10: F1 vacia
  BSPG10: G1 vacia
  BSPH10: H1 vacia
  BSPA20: A2 vacia
  BSPB20: B2 vacia
  BSPC20: C2 vacia
  BSPD20: D2 vacia
  BSPE20: E2 vacia
  BSPF20: F2 vacia
  BSPG20: G2 vacia
  BSPH20: H2 vacia
  BSPA30: A3 vacia
  BSPB30: B3 vacia
  BSPC30: C3 vacia
  BSPD30: D3 vacia
  BSPE30: E3 vacia
  BSPF30: F3 vacia
  BSPG30: G3 vacia
  BSPH30: H3 vacia
  BSPA40: A4 vacia
  BSPB40: B4 vacia
  BSPC40: C4 vacia
  BSPF40: F4 vacia
  BSPG40: G4 vacia
  BSPH40: H4 vacia
  BSPA50: A5 vacia
  BSPB50: B5 vacia
  BSPC50: C5 vacia
  BSPF50: F5 vacia
  BSPG50: G5 vacia
